In [ ]:
# Step 1: Census Geocoder — resolves free-text address → lat/lng + structured address
# Same TIGER/Line data the FCC uses. No API key. US-only.

import urllib.request, urllib.parse, json, time, math

CENSUS_URL = 'https://geocoding.geo.census.gov/geocoder/locations/onelineaddress'

TEST_ADDRESSES = [
    '1494 Main St Cincinnati OH',
    '2722 Burnet Ave Cincinnati OH 45219',
    '123 Main St Cincinnati OH',
    '550 Main St Cincinnati OH 45202',
    '301 Congress Ave Austin TX 78701',
    '350 5th Ave New York NY 10001',
    # Ambiguous / short — should gracefully miss
    '1494',
    'Cincinnati OH',
]

def census_geocode(address):
    params = urllib.parse.urlencode({
        'address':   address,
        'benchmark': 'Public_AR_Current',
        'format':    'json',
    })
    url = f'{CENSUS_URL}?{params}'
    req = urllib.request.Request(url, headers={'User-Agent': 'Internet-4-ALL/test'})
    try:
        with urllib.request.urlopen(req, timeout=10) as r:
            data = json.loads(r.read())
    except Exception as e:
        return url, None, str(e)

    matches = data.get('result', {}).get('addressMatches', [])
    if not matches:
        return url, None, 'no match'

    m      = matches[0]
    coords = m.get('coordinates', {})
    addr   = m.get('addressComponents', {})
    return url, {
        'matchedAddress': m.get('matchedAddress', ''),
        'lat':            float(coords.get('y', 0)),
        'lng':            float(coords.get('x', 0)),
        'streetNumber':   addr.get('fromAddress', ''),
        'streetName':     addr.get('streetName', ''),
        'suffix':         addr.get('suffixType', ''),
        'city':           addr.get('city', ''),
        'state':          addr.get('state', ''),
        'zip':            addr.get('zip', ''),
    }, None

print('Census geocoder fn ready')

In [ ]:
print('\n' + '='*80)
print('TEST 1 — Census Geocoder')
print('='*80)

census_results = {}
for addr in TEST_ADDRESSES:
    url, result, err = census_geocode(addr)
    if err:
        print(f'  MISS  "{addr}"  -> {err}')
        print(f'         URL: {url}')
    else:
        street_comp = f"#{result['streetNumber']} {result['streetName']} {result['suffix']}".strip()
        print(f'  HIT   "{addr}"')
        print(f'         -> {result["matchedAddress"]}')
        print(f'            lat={result["lat"]}  lng={result["lng"]}  zip={result["zip"]}')
        print(f'            components: {street_comp}, {result["city"]}, {result["state"]}')
    census_results[addr] = result
    print()

print('Done.')

In [ ]:
# Step 2: FCC API helpers
# Tests both header variants (plain vs FCC Referer) because the FCC API
# is known to gate non-browser origins with status_code 405.

FCC_BASE = 'https://broadbandmap.fcc.gov/api/public/map'

FCC_HEADERS_PLAIN = {
    'Accept':     'application/json',
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36',
}
FCC_HEADERS_REFERER = {
    **FCC_HEADERS_PLAIN,
    'Origin':  'https://broadbandmap.fcc.gov',
    'Referer': 'https://broadbandmap.fcc.gov/',
}

def fcc_list_availability(lat, lng, addr, city, state, zipcode, headers):
    params = urllib.parse.urlencode({
        'latitude': lat, 'longitude': lng, 'unit': '',
        'category': 'Fixed Broadband',
        'addr': addr, 'city': city, 'state': state, 'zip': zipcode,
    })
    url = f'{FCC_BASE}/listAvailability?{params}'
    req = urllib.request.Request(url, headers=headers)
    try:
        with urllib.request.urlopen(req, timeout=12) as r:
            data = json.loads(r.read())
        providers   = data.get('availability') or data.get('providers') or []
        location_id = data.get('location_id')
        status_code = data.get('status_code')
        return url, providers, location_id, status_code, None
    except Exception as e:
        return url, [], None, None, str(e)

def fcc_location_search(lat, lng, addr, city, state, zipcode, headers):
    params = urllib.parse.urlencode({
        'latitude': lat, 'longitude': lng,
        'addr': addr, 'city': city, 'state': state, 'zip': zipcode, 'unit': '',
    })
    url = f'{FCC_BASE}/location/search?{params}'
    req = urllib.request.Request(url, headers=headers)
    try:
        with urllib.request.urlopen(req, timeout=10) as r:
            data = json.loads(r.read())
        locations = data.get('data') or data.get('locations') or []
        if isinstance(locations, list) and locations:
            loc     = locations[0]
            loc_id  = loc.get('location_id') or loc.get('id')
            matched = loc.get('address_full') or loc.get('addr') or str(loc)[:80]
            return url, loc_id, matched, None
        return url, None, f'0 results (keys: {list(data.keys())})', None
    except Exception as e:
        return url, None, None, str(e)

def fcc_by_location_id(location_id, headers):
    url = f'{FCC_BASE}/availability/location?location_id={location_id}'
    req = urllib.request.Request(url, headers=headers)
    try:
        with urllib.request.urlopen(req, timeout=12) as r:
            data = json.loads(r.read())
        providers = (data.get('data') or {}).get('providers') or \
                    data.get('providers') or data.get('availability') or []
        return url, providers, None
    except Exception as e:
        return url, [], str(e)

def print_providers(providers, limit=6):
    for i, p in enumerate(providers[:limit]):
        name = p.get('brand_name') or p.get('dba_name') or p.get('holding_company_name') or '?'
        tech = p.get('technology_name') or p.get('tech_code') or '?'
        dl   = p.get('max_advertised_download_speed') or p.get('max_download_mbps') or '?'
        print(f'    {i+1}. {name:<32} | {str(tech):<20} | down {dl} Mbps')
    if len(providers) > limit:
        print(f'    ... and {len(providers)-limit} more')

print('FCC helpers ready')

In [ ]:
FCC_TEST_ADDRS = [
    '1494 Main St Cincinnati OH',
    '550 Main St Cincinnati OH 45202',
    '301 Congress Ave Austin TX 78701',
]

print('\n' + '='*80)
print('TEST 2 — FCC listAvailability (Census-resolved coords + address components)')
print('='*80)

for query in FCC_TEST_ADDRS:
    r = census_results.get(query)
    if not r:
        print(f'  SKIP (no Census result): {query}')
        continue

    street = f"{r['streetNumber']} {r['streetName']} {r['suffix']}".strip()
    print(f'\n  [{query}]')
    print(f'  Census -> {r["matchedAddress"]}  lat={r["lat"]}  lng={r["lng"]}')

    for label, hdrs in [('plain', FCC_HEADERS_PLAIN), ('+Referer', FCC_HEADERS_REFERER)]:
        url, providers, loc_id, api_code, err = fcc_list_availability(
            r['lat'], r['lng'], street, r['city'], r['state'], r['zip'], hdrs
        )
        if err:
            print(f'  [{label}] ERROR: {err}')
        elif not providers:
            print(f'  [{label}] 0 providers  location_id={loc_id}  api_status_code={api_code}')
            print(f'            {url}')
        else:
            print(f'  [{label}] {len(providers)} providers  location_id={loc_id}')
            print_providers(providers)

print('\nDone.')

In [ ]:
print('\n' + '='*80)
print('TEST 3 — FCC location/search -> location_id -> availability/location')
print('='*80)

for query in FCC_TEST_ADDRS:
    r = census_results.get(query)
    if not r:
        print(f'  SKIP: {query}')
        continue

    street = f"{r['streetNumber']} {r['streetName']} {r['suffix']}".strip()
    print(f'\n  [{query}]')

    for label, hdrs in [('plain', FCC_HEADERS_PLAIN), ('+Referer', FCC_HEADERS_REFERER)]:
        url, loc_id, matched, err = fcc_location_search(
            r['lat'], r['lng'], street, r['city'], r['state'], r['zip'], hdrs
        )
        if err:
            print(f'  [{label}] location/search ERROR: {err}')
            continue
        if not loc_id:
            print(f'  [{label}] location/search: {matched}')
            print(f'            {url}')
            continue

        print(f'  [{label}] location/search OK: location_id={loc_id}  matched="{matched}"')

        url2, providers, err2 = fcc_by_location_id(loc_id, hdrs)
        if err2:
            print(f'  [{label}] availability/location ERROR: {err2}')
        elif not providers:
            print(f'  [{label}] availability/location: 0 providers  {url2}')
        else:
            print(f'  [{label}] availability/location: {len(providers)} providers')
            print_providers(providers)

print('\nDone.')

In [ ]:
# TEST 4: broadbandmap.com (current production) vs FCC official — side by side
# Key question: does FCC official work from a non-browser origin at all?
# If blocked -> broadbandmap.com stays, but Census Geocoder still helps.
# If unblocked -> switch to FCC location_id path for unit-level accuracy.

print('\n' + '='*80)
print('TEST 4 — broadbandmap.com (current) vs FCC official — side by side')
print('='*80)

def bbmap_providers(lat, lng):
    url = f'https://broadbandmap.com/api/v1/location/internet?lat={lat}&lng={lng}'
    req = urllib.request.Request(url, headers={'User-Agent': 'Internet-4-ALL/test'})
    try:
        with urllib.request.urlopen(req, timeout=10) as r:
            data = json.loads(r.read())
        return data.get('providers', []), None
    except Exception as e:
        return [], str(e)

for query in FCC_TEST_ADDRS:
    r = census_results.get(query)
    if not r:
        continue

    print(f'\n  {query}')
    print(f'  lat={r["lat"]}  lng={r["lng"]}')

    bb, err = bbmap_providers(r['lat'], r['lng'])
    if err:
        print(f'  broadbandmap.com  ERROR: {err}')
    else:
        names = [p.get('name', '?') for p in bb]
        print(f'  broadbandmap.com  {len(bb)} providers: {names}')

    street = f"{r['streetNumber']} {r['streetName']} {r['suffix']}".strip()
    _, fcc, _, _, err2 = fcc_list_availability(
        r['lat'], r['lng'], street, r['city'], r['state'], r['zip'], FCC_HEADERS_PLAIN
    )
    if err2:
        print(f'  FCC official      ERROR: {err2}')
    else:
        fcc_names = [p.get('brand_name') or p.get('dba_name', '?') for p in fcc]
        status = 'OK' if fcc else 'BLOCKED/EMPTY'
        print(f'  FCC official      {status}  {len(fcc)} providers: {fcc_names}')

print()
print('--- CONCLUSION ---')
print('FCC official 0/blocked -> keep broadbandmap.com + add Census Geocoder for address resolution')
print('FCC official works     -> switch to FCC location/search -> availability/location (unit-level accuracy)')